In [1]:
import shutil, os
from pathlib import Path

# 1. Setup Repo (Branch: feature/llama-4-scout)
if Path('/content/pulmovision').exists(): shutil.rmtree('/content/pulmovision')
!git clone --branch feature/llama-4-scout https://github.com/HarithHadi/PulmoVision /content/pulmovision
%cd /content/pulmovision/evaluation

# 2. Install Dependencies
!pip install -q rouge-score nltk scikit-learn matplotlib kaggle groq fastapi uvicorn transformers torch torchvision peft bitsandbytes accelerate sentencepiece open-clip-torch opencv-python-headless python-multipart pyngrok nest_asyncio supabase bcrypt python-jose[cryptography] python-dotenv

# 3. Data Setup (Mount Drive & Download Dataset)
from google.colab import drive
drive.mount('/content/drive')
# Ensure KAGGLE_USERNAME and KAGGLE_API_TOKEN are set in Colab Secrets
!mkdir -p /root/.kaggle && echo '{"username":"'${KAGGLE_USERNAME}'","key":"'${KAGGLE_API_TOKEN}'"}' > /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d raddar/chest-xrays-indiana-university
!unzip -q chest-xrays-indiana-university.zip -d indiana

Cloning into '/content/pulmovision'...
remote: Enumerating objects: 477, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 477 (delta 11), reused 28 (delta 7), pack-reused 437 (from 1)
Receiving objects: 100% (477/477), 6.55 MiB | 10.68 MiB/s, done.
Resolving deltas: 100% (247/247), done.
[Errno 2] No such file or directory: '/content/pulmovision/evaluation'
/content
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset URL: https://www.kaggle.com/datasets/raddar/chest-xrays-indiana-university
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
Resuming from 1270874112 bytes (12863639843 bytes left)...
100% 13.2G/13.2G [02:15<00:00, 94.8MB/s]



In [5]:
%%writefile evaluate_reports.py
import pandas as pd
from rouge_score import rouge_scorer
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

def evaluate_reports(csv_path="indiana/indiana_reports.csv"):
    df = pd.read_csv(csv_path).dropna(subset=['impression'])
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    results = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating"):
        ref, gen = str(row['impression']), str(row['impression']) # Placeholder logic
        scores = scorer.score(ref, gen)
        for key in results: results[key].append(scores[key].fmeasure)
    
    means = [np.mean(results['rouge1']), np.mean(results['rouge2']), np.mean(results['rougeL'])]
    
    # Plotting
    plt.figure(figsize=(8, 5))
    plt.bar(['ROUGE-1', 'ROUGE-2', 'ROUGE-L'], means, color=['skyblue', 'salmon', 'lightgreen'])
    plt.ylim(0, 1.1)
    plt.ylabel('F-Measure')
    plt.title('Report Evaluation Metrics')
    for i, v in enumerate(means): plt.text(i, v + 0.02, f"{v:.4f}", ha='center')
    plt.savefig("rouge_scores.png")
    print("Saved rouge_scores.png")

if __name__ == "__main__":
    evaluate_reports()

Overwriting evaluate_reports.py


In [6]:
# This runs the code you just wrote to the file
!python3 evaluate_reports.py

Evaluating: 100% 3820/3820 [00:02<00:00, 1663.71it/s]
Saved rouge_scores.png


In [16]:
!find /content/pulmovision -name "tb_classifier*.pt"

In [13]:
# 5. CLASSIFIER EVALUATION
import torch
import numpy as np
import os # <--- Added missing import
from pathlib import Path # <--- Added missing import
from PIL import Image
from torchvision import transforms
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
import matplotlib.pyplot as plt
from tqdm import tqdm
import sys

# 1. Update the base path and working directory
repo_root = Path('/content/pulmovision')
sys.path.append(str(repo_root))
os.chdir(repo_root)

from backend.dependencies import ModelContainer

models = ModelContainer()
models.load_tb_model()
device = "cuda" if torch.cuda.is_available() else "cpu"

preprocess = transforms.Compose([
    transforms.Resize((518, 518)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

y_true, y_scores, y_pred = [], [], []
dataset_root = Path("indiana") 
# Check that this path actually exists before looping
folders = [(1, dataset_root / "images" / "Tuberculosis"), (0, dataset_root / "images" / "Normal")]

for label, folder in folders:
    if not folder.exists(): 
        print(f"Warning: Folder not found: {folder}")
        continue
    image_files = list(folder.glob("*.png"))
    for img_path in tqdm(image_files, desc=f"Processing {folder.name}"):
        image = Image.open(img_path).convert("RGB")
        tensor = preprocess(image).unsqueeze(0).to(device)
        results = models.tb_classifier.run_pipeline(tensor)
        prob = float(results["probs"][1])
        y_scores.append(prob)
        y_true.append(label)
        y_pred.append(1 if prob >= 0.5 else 0)

# Calculate metrics
auc = roc_auc_score(y_true, y_scores)
print(f"\nAUC-ROC: {auc:.4f}")
print(classification_report(y_true, y_pred, target_names=["Normal", "TB"]))

# Plot ROC curve
fpr, tpr, _ = roc_curve(y_true, y_scores)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color="blue", label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--") # <--- Baseline for reference
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — PulmoVision TB Classifier")
plt.legend()
plt.savefig("roc_curve.png")
print("Saved roc_curve.png")

Loading TB classifier once...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

FileNotFoundError: [Errno 2] No such file or directory: 'models/tb_classifier (5).pt'